# Part 4 Task 1 — Variational Autoencoder on the OASIS brain MRI dataset

A convolutional VAE trained on preprocessed OASIS MR brain slices, plus a visualisation of the
latent manifold it learns. The manifold visualisation is what the marks hinge on, so stage 4
produces **both** accepted forms:

- **(a) a sampling grid** — decode a 2D sweep through latent space into one large image
- **(b) a dimensionality-reduction scatter** — encode the test set, project to 2D, plot it

**Staged structure** — every stage is gated by an environment variable so the cheap ones run alone:

| stage | what it does | needs data? | needs GPU? | cost |
|---|---|---|---|---|
| 1 | Architecture sanity check — random noise in, check shapes and parameter count | no | no | seconds |
| 2 | Smoke test — a handful of real images, 2 epochs, loss drops, show a reconstruction | yes | no | ~1 min |
| 3 | Full training run, timed, with total / reconstruction / KL loss curves | yes | yes | minutes |
| 4 | Manifold visualisation (both forms) | yes | yes | ~1 min |
| 5 | Save reconstructions and samples as PNGs for the SLURM run to leave behind | yes | yes | seconds |

Stage 1 always runs. Stages 2–5 are controlled by `VAE_STAGE2` … `VAE_STAGE5`.

## Configuration

### Why downsample from 256×256?

The OASIS slices are 256×256. A convolutional VAE at that resolution is roughly **16× the pixels**
of 64×64, which costs proportionally more memory and time per epoch, and forces either a small
batch size or a narrow network. None of that buys much here: these are registered, skull-stripped
brain slices that vary smoothly and mostly at coarse scale, so the structure a VAE can actually
model survives downsampling. `IMG_SIZE = 64` trains in minutes and leaves a clean manifold;
128 also works if you want sharper reconstructions and have the GPU time.

`IMG_SIZE` must be divisible by 16, because the encoder has four stride-2 blocks.

### Why `LATENT_DIM = 16` rather than 2?

There is a real tension here:

- **`LATENT_DIM = 2`** lets you sweep the latent space *directly* — the textbook manifold grid.
  But two numbers is a severe bottleneck, so reconstructions come out blurry and averaged.
- **`LATENT_DIM = 16`** reconstructs well, but you cannot sweep 16 axes on a 2D page.

The default is 16, and the grid in stage 4 handles it by sweeping the **two principal directions
of the encoded training set** — still a genuine 2D plane through the learned manifold, just not
axis-aligned. If you prefer the classic direct sweep, set `VAE_LATENT=2` and the code
automatically switches to sweeping the raw axes instead. Both paths are implemented.

In [ ]:
import os
import re
import time
import json
import glob
import platform

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from PIL import Image

import matplotlib
matplotlib.use("Agg")      # non-interactive backend so figures save cleanly under nbconvert
import matplotlib.pyplot as plt


def env_int(name, default):
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name, default):
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else default


def env_flag(name, default=False):
    raw = os.environ.get(name, "").strip().lower()
    return raw in ("1", "true", "yes", "y", "on") if raw else default


# ---------------------------------------------------------------- device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True   # autotune convs for our fixed input size

SEED = env_int("VAE_SEED", 42)
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------- data location
# Read-only group directory on Rangpur. Never copied, never written to.
OASIS_ROOT = os.environ.get("VAE_DATA", "/home/groups/comp3710/OASIS")
TRAIN_DIR = os.path.join(OASIS_ROOT, "keras_png_slices_train")
VAL_DIR = os.path.join(OASIS_ROOT, "keras_png_slices_validate")
TEST_DIR = os.path.join(OASIS_ROOT, "keras_png_slices_test")
# NOTE: the keras_png_slices_seg_* folders are segmentation masks for a different task.
# We never touch them — globbing the three directories above picks up only the MR images.

DATA_AVAILABLE = os.path.isdir(TRAIN_DIR)

# ---------------------------------------------------------------- hyperparameters
IMG_SIZE = env_int("VAE_IMG", 64)        # must be divisible by 16
LATENT_DIM = env_int("VAE_LATENT", 16)   # set to 2 for the classic direct-sweep manifold
BATCH_SIZE = env_int("VAE_BATCH", 128)
EPOCHS = env_int("VAE_EPOCHS", 30)
LR = env_float("VAE_LR", 1e-3)

# BETA weights the KL term against reconstruction. 1.0 is the true variational bound.
BETA = env_float("VAE_BETA", 1.0)
# Linearly ramp beta from 0 to BETA over this many epochs. Guards against posterior collapse,
# where the KL term wins early, the encoder outputs the prior for every input, and the decoder
# learns to ignore z entirely. See the loss cell for the full explanation.
KL_WARMUP_EPOCHS = env_int("VAE_KL_WARMUP", 10)

NUM_WORKERS = env_int("VAE_WORKERS", 0 if platform.system() == "Windows" else 4)

# ---------------------------------------------------------------- stage gates
RUN_STAGE2 = env_flag("VAE_STAGE2", True)    # cheap, on by default
RUN_STAGE3 = env_flag("VAE_STAGE3", False)   # full training
RUN_STAGE4 = env_flag("VAE_STAGE4", False)   # manifold visualisation
RUN_STAGE5 = env_flag("VAE_STAGE5", False)   # save PNG artifacts

OUT_DIR = os.environ.get("VAE_OUT", "./vae_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

assert IMG_SIZE % 16 == 0, f"IMG_SIZE must be divisible by 16 (four stride-2 blocks), got {IMG_SIZE}"

print(f"device        : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"torch         : {torch.__version__}")
print(f"data root     : {OASIS_ROOT}  (available: {DATA_AVAILABLE})")
print(f"image size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"latent dim    : {LATENT_DIM}")
print(f"batch / epochs: {BATCH_SIZE} / {EPOCHS}")
print(f"beta          : {BETA}  (warmup {KL_WARMUP_EPOCHS} epochs)")
print(f"stages 2/3/4/5: {RUN_STAGE2}/{RUN_STAGE3}/{RUN_STAGE4}/{RUN_STAGE5}")
print(f"output dir    : {OUT_DIR}")

## The dataset

A plain `torch.utils.data.Dataset` that reads the PNGs straight off the read-only group
directory. Three methods is all a Dataset needs: `__init__` to build an index, `__len__`, and
`__getitem__` to load and transform one sample.

Deliberate choices:

- **Load lazily in `__getitem__`, not up front.** 9,664 images at 256×256 float32 would be about
  2.5 GB resident. Loading per item keeps memory flat and lets the DataLoader workers overlap
  disk I/O with GPU compute.
- **`.convert("L")`** forces single-channel 8-bit greyscale. The files are already mode `L`, but
  being explicit means a stray RGB or 16-bit file cannot silently change the tensor shape.
- **No augmentation.** These slices are spatially registered — flipping or rotating them would
  destroy the anatomical alignment the VAE is meant to learn. This is the opposite of the CIFAR
  situation in part 3.2.
- **Scaled to [0, 1]**, which is what the sigmoid output and the BCE reconstruction loss expect.
- **Slice index parsed from the filename.** Files are named `case_XXX_slice_YY.nii.png`. Keeping
  `YY` gives us something meaningful to colour the stage-4 scatter by: if the VAE has learned
  anything anatomical, points should organise by position through the brain.

In [ ]:
class OASISDataset(Dataset):
    """OASIS preprocessed MR brain slices, read directly from the group directory.

    Returns (image, slice_index) where image is a float32 tensor of shape
    (1, IMG_SIZE, IMG_SIZE) scaled to [0, 1].

    The slice index is metadata only — the VAE is unsupervised and never sees it during
    training. It is used purely to colour the latent scatter plot in stage 4.
    """

    # Matches the trailing slice number in e.g. "case_001_slice_10.nii.png"
    _SLICE_RE = re.compile(r"slice_(\d+)")

    def __init__(self, root, img_size=64, limit=None):
        self.root = root
        self.img_size = img_size

        # Sort so the ordering is deterministic across runs and machines — glob order is
        # filesystem-dependent, which would silently break reproducibility.
        self.paths = sorted(glob.glob(os.path.join(root, "*.png")))
        if limit is not None:
            self.paths = self.paths[:limit]

        if not self.paths:
            raise FileNotFoundError(f"no PNGs found in {root}")

        # Parse the slice number once, at construction, rather than per __getitem__ call.
        self.slice_idx = []
        for p in self.paths:
            m = self._SLICE_RE.search(os.path.basename(p))
            self.slice_idx.append(int(m.group(1)) if m else -1)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        # "L" = 8-bit single-channel greyscale.
        img = Image.open(self.paths[i]).convert("L")

        # BILINEAR is the right filter for downsampling continuous-valued medical intensities;
        # NEAREST would alias badly. (NEAREST would be correct for the segmentation masks,
        # where interpolating between label values is meaningless — but we do not use those.)
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        # uint8 [0,255] -> float32 [0,1], then add the channel axis: (H, W) -> (1, H, W).
        x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).unsqueeze(0)

        return x, self.slice_idx[i]


def make_loader(root, batch_size, shuffle, limit=None):
    """Build a DataLoader over one OASIS split."""
    ds = OASISDataset(root, img_size=IMG_SIZE, limit=limit)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),      # faster host->GPU copies
        persistent_workers=(NUM_WORKERS > 0),    # do not respawn workers every epoch
        drop_last=shuffle,                       # only drop the ragged last batch when training
    )
    return ds, loader


if DATA_AVAILABLE:
    _probe = OASISDataset(TRAIN_DIR, img_size=IMG_SIZE, limit=8)
    _x, _s = _probe[0]
    print(f"train images found : {len(glob.glob(os.path.join(TRAIN_DIR, '*.png'))):,}")
    print(f"sample tensor shape: {tuple(_x.shape)}  dtype {_x.dtype}")
    print(f"sample value range : [{_x.min():.3f}, {_x.max():.3f}]")
    print(f"sample slice index : {_s}")
else:
    print(f"Data not found at {TRAIN_DIR}")
    print("Stage 1 will still run (it needs no data). Stages 2-5 will skip.")

## The model

### Why the encoder outputs `mu` and `logvar` instead of one vector

A plain autoencoder maps each image to a single point in latent space. Nothing constrains the
space *between* those points, so decoding an arbitrary `z` usually gives noise — the space has
holes and the model cannot generate.

A VAE instead maps each image to a **distribution**: a Gaussian with mean `mu` and diagonal
covariance. Every input claims a small region of latent space rather than a single point, and
because the KL term pulls all those regions toward a shared unit Gaussian, they overlap and tile
the space continuously. That continuity is exactly what makes the stage-4 manifold sweep produce
meaningful images instead of static.

### Why `logvar` and not `var` or `sigma`

Variance must be strictly positive. If the network predicted it directly, we would need to clamp
or exponentiate the output anyway, and a near-zero prediction would blow up the `log` in the KL
term. Predicting the **log** of the variance means the raw network output is unconstrained — any
real number is valid — and `sigma = exp(0.5 * logvar)` is positive by construction. It is also
numerically better behaved, since the KL formula needs `log(var)` directly.

### The reparameterisation trick

We need to sample `z ~ N(mu, sigma^2)` and then backpropagate through that sampling step to train
the encoder. But sampling is not differentiable: there is no derivative of "draw a random number"
with respect to `mu`.

The trick moves the randomness out of the path that needs gradients:

```
    z = mu + sigma * eps ,    eps ~ N(0, I)
```

`eps` is drawn from a *fixed* distribution with no parameters, so it is just a constant as far as
autograd is concerned. `z` is now a smooth, differentiable function of `mu` and `sigma`, and
gradients flow back into the encoder — while `z` still has exactly the distribution we wanted.
This is the single idea that makes a VAE trainable end to end.

### Architecture

Four stride-2 conv blocks down, a symmetric four transposed-conv blocks back up. At
`IMG_SIZE = 64`: 64 → 32 → 16 → 8 → 4, with channels 1 → 32 → 64 → 128 → 256.

In [ ]:
class VAE(nn.Module):
    """Convolutional VAE for single-channel square images.

    encode: (B, 1, S, S) -> mu, logvar, each (B, latent_dim)
    decode: (B, latent_dim) -> (B, 1, S, S) in [0, 1]
    """

    def __init__(self, img_size=64, latent_dim=16, base_ch=32):
        super().__init__()
        self.img_size = img_size
        self.latent_dim = latent_dim

        # Four stride-2 blocks, so the feature map is img_size / 16 on a side.
        self.feat_size = img_size // 16
        self.feat_ch = base_ch * 8              # 256 when base_ch=32
        flat = self.feat_ch * self.feat_size ** 2

        # ---------------- encoder ----------------
        # Each block halves the spatial size and doubles the channels. BatchNorm keeps
        # activations well-scaled; bias=False because BatchNorm's shift subsumes it.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_ch, 4, stride=2, padding=1, bias=False),          # S -> S/2
            nn.BatchNorm2d(base_ch), nn.ReLU(inplace=True),

            nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1, bias=False),  # S/2 -> S/4
            nn.BatchNorm2d(base_ch * 2), nn.ReLU(inplace=True),

            nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1, bias=False),  # S/4 -> S/8
            nn.BatchNorm2d(base_ch * 4), nn.ReLU(inplace=True),

            nn.Conv2d(base_ch * 4, base_ch * 8, 4, stride=2, padding=1, bias=False),  # S/8 -> S/16
            nn.BatchNorm2d(base_ch * 8), nn.ReLU(inplace=True),
        )

        # TWO separate heads off the same shared features. This is the structural difference
        # from a plain autoencoder: one vector describing where the distribution sits, one
        # describing how wide it is.
        self.fc_mu = nn.Linear(flat, latent_dim)
        self.fc_logvar = nn.Linear(flat, latent_dim)

        # ---------------- decoder ----------------
        self.fc_decode = nn.Linear(latent_dim, flat)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(self.feat_ch, base_ch * 4, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 4), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 2), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(base_ch, 1, 4, stride=2, padding=1),
            # Sigmoid squashes the output to [0, 1] to match our normalised inputs, and is
            # what makes binary cross-entropy a valid reconstruction loss.
            nn.Sigmoid(),
        )

    def encode(self, x):
        """(B, 1, S, S) -> mu, logvar"""
        h = self.encoder(x)
        h = torch.flatten(h, 1)              # (B, feat_ch * feat_size^2)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        """Sample z ~ N(mu, sigma^2) differentiably.

        logvar = log(sigma^2)  =>  sigma = exp(0.5 * logvar)
        """
        std = torch.exp(0.5 * logvar)
        # randn_like draws from N(0, I) with no learnable parameters — autograd treats it
        # as a constant, which is precisely why the gradient can pass through to mu and std.
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        """(B, latent_dim) -> (B, 1, S, S)"""
        h = self.fc_decode(z)
        # Undo the flatten: back to a (B, C, H, W) feature map the transposed convs can grow.
        h = h.view(-1, self.feat_ch, self.feat_size, self.feat_size)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar

## The loss

A VAE maximises the **evidence lower bound (ELBO)**, which in the form we minimise is two terms:

```
  loss  =  reconstruction  +  beta * KL
```

### Reconstruction term

How well does the decoded image match the input? We use binary cross-entropy summed over pixels.
BCE is the natural match for a sigmoid output on [0,1]-scaled data and penalises confident
mistakes harder than MSE does, which in practice gives less washed-out reconstructions.

We sum over pixels and average over the batch, rather than averaging over pixels. This matters:
the KL term is inherently a per-image sum over latent dimensions, so if the reconstruction term
were a per-pixel *mean* the two would be on wildly different scales (4096 pixels vs 16 latents)
and `beta = 1` would not mean what the theory says it means.

### KL term

`KL( N(mu, sigma^2) || N(0, I) )` measures how far each image's encoded distribution has drifted
from the standard normal prior. For diagonal Gaussians it has a closed form:

```
  KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
```

It is a **regulariser on the shape of the latent space**. Without it the encoder would scatter
images anywhere and shrink every sigma to zero — reverting to a plain autoencoder with a
discontinuous, un-sampleable space. The KL pulls every posterior toward the same unit Gaussian,
which is what lets us sample `z ~ N(0, I)` in stage 4 and get a plausible brain back.

### Why the two are weighted against each other

They pull in opposite directions. Reconstruction wants each image to occupy its own distinct,
tight region so it can be reproduced exactly. KL wants every image to map to the *same*
distribution. The balance decides what you get:

- **KL too strong** → *posterior collapse*: the encoder outputs the prior for every input, the
  decoder ignores `z` entirely, and every sample decodes to the same blurry average brain.
- **KL too weak** → the space becomes discontinuous and full of holes; reconstructions are sharp
  but sampling and interpolation produce garbage, so the manifold plot is meaningless.

`beta = 1.0` is the true ELBO. Above 1 (a *beta-VAE*) buys more disentangled, smoother latents at
the cost of blurrier reconstructions.

### KL warm-up

Early in training the decoder is useless, so the cheapest way to cut the loss is to collapse the
KL to zero — and a model that starts in collapse rarely escapes. Ramping `beta` linearly from 0
over the first `KL_WARMUP_EPOCHS` lets the decoder become useful before the regulariser bites.

In [ ]:
def vae_loss(recon, x, mu, logvar, beta=1.0):
    """Return (total, reconstruction, kl) — all per-image averages over the batch."""
    batch = x.size(0)

    # Sum over all pixels of all images, then divide by batch size => per-image sum.
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum") / batch

    # Closed-form KL between N(mu, sigma^2) and N(0, I), summed over latent dims,
    # averaged over the batch.
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch

    return recon_loss + beta * kl, recon_loss, kl


def beta_at_epoch(epoch, total_warmup=KL_WARMUP_EPOCHS, beta_max=BETA):
    """Linear KL warm-up: 0 -> beta_max over `total_warmup` epochs, then constant."""
    if total_warmup <= 0:
        return beta_max
    return beta_max * min(1.0, epoch / total_warmup)

## Stage 1 — architecture sanity check

No data, no GPU. Push random noise of the right shape through the model and verify the output
shape matches the input, that `mu`/`logvar` are the declared width, and that gradients reach
every parameter. Catches essentially every architecture bug — a mismatched `feat_size`, a
transposed-conv stack that does not land back on the input resolution — in about two seconds.

In [ ]:
print("=" * 64)
print("STAGE 1 — architecture sanity check (random noise, no data)")
print("=" * 64)

model_check = VAE(img_size=IMG_SIZE, latent_dim=LATENT_DIM)
model_check.eval()

# torch.rand (uniform [0,1]) rather than torch.randn (normal, unbounded): real inputs are
# scaled to [0,1], and the BCE reconstruction loss below *requires* its target in that range.
# randn here would raise "all elements of target should be between 0 and 1".
dummy = torch.rand(4, 1, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    recon, mu, logvar = model_check(dummy)

print(f"input  shape : {tuple(dummy.shape)}")
print(f"recon  shape : {tuple(recon.shape)}")
print(f"mu     shape : {tuple(mu.shape)}")
print(f"logvar shape : {tuple(logvar.shape)}")
print(f"recon range  : [{recon.min():.4f}, {recon.max():.4f}]  (sigmoid => must be within [0,1])")

# The defining contract of an autoencoder: output shape == input shape.
assert recon.shape == dummy.shape, f"recon {tuple(recon.shape)} != input {tuple(dummy.shape)}"
assert mu.shape == (4, LATENT_DIM), f"mu has shape {tuple(mu.shape)}"
assert logvar.shape == (4, LATENT_DIM), f"logvar has shape {tuple(logvar.shape)}"
assert 0.0 <= recon.min() and recon.max() <= 1.0, "sigmoid output escaped [0,1]"

# Intermediate encoder feature map, to confirm the four stride-2 blocks landed where we think.
with torch.no_grad():
    feat = model_check.encoder(dummy)
print(f"encoder feature map: {tuple(feat.shape)}  "
      f"(expected (4, {model_check.feat_ch}, {model_check.feat_size}, {model_check.feat_size}))")
assert feat.shape[2] == model_check.feat_size, "encoder downsampling does not match feat_size"

n_params = sum(p.numel() for p in model_check.parameters() if p.requires_grad)
print(f"\ntrainable parameters: {n_params:,}")

# --- gradients reach everything ---
model_check.train()
recon, mu, logvar = model_check(dummy)
total, rec, kl = vae_loss(recon, dummy, mu, logvar, beta=1.0)
total.backward()
missing = [n for n, p in model_check.named_parameters() if p.requires_grad and p.grad is None]
assert not missing, f"parameters with no gradient: {missing[:5]}"
print(f"loss on random input: total {total.item():.2f}  recon {rec.item():.2f}  kl {kl.item():.2f}")

print("\nSTAGE 1 PASSED — shapes, output range and gradient flow all correct.")
del model_check

## Training and evaluation functions

Shared by the smoke test and the full run, so stage 2 exercises exactly the code stage 3 uses.

In [ ]:
def train_one_epoch(model, loader, optimizer, beta):
    """One pass over `loader`. Returns per-image mean (total, recon, kl)."""
    model.train()
    tot = rec_tot = kl_tot = 0.0
    n = 0

    for x, _ in loader:                      # the slice index is metadata; training ignores it
        x = x.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        recon, mu, logvar = model(x)
        loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)

        loss.backward()
        optimizer.step()

        # Weight by batch size so a ragged final batch does not skew the epoch mean.
        b = x.size(0)
        tot += loss.item() * b
        rec_tot += rec.item() * b
        kl_tot += kl.item() * b
        n += b

    return tot / n, rec_tot / n, kl_tot / n


@torch.no_grad()
def evaluate(model, loader, beta):
    """Same metrics on a held-out split, with autograd off."""
    model.eval()
    tot = rec_tot = kl_tot = 0.0
    n = 0

    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        recon, mu, logvar = model(x)
        loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)

        b = x.size(0)
        tot += loss.item() * b
        rec_tot += rec.item() * b
        kl_tot += kl.item() * b
        n += b

    return tot / n, rec_tot / n, kl_tot / n


def show_grid(images, nrow, path, title=None, figsize=(8, 8)):
    """Tile a (N, 1, S, S) tensor into one image and save it to `path`."""
    imgs = images.detach().cpu().numpy()
    n = imgs.shape[0]
    ncol = int(np.ceil(n / nrow))

    fig, axes = plt.subplots(nrow, ncol, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i, 0], cmap="gray", vmin=0, vmax=1)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    print(f"saved -> {path}")
    plt.show()
    plt.close(fig)

## Stage 2 — tiny smoke test

Real OASIS images, but only a few hundred of them for 2 epochs. This is not trying to learn
anything useful. It answers three cheap questions before committing to a GPU queue:

1. Does the Dataset actually read the PNGs and produce `(B, 1, S, S)` float tensors in [0, 1]?
2. Does the loss go **down**, i.e. is the optimiser correctly wired?
3. Does anything produce NaN or inf?

It also displays one input/reconstruction pair. After 2 epochs the reconstruction will be a
blurry blob — that is expected and fine. What matters is that it is a *brain-shaped* blur rather
than noise, which confirms the whole pipeline end to end.

In [ ]:
print("=" * 64)
print("STAGE 2 — tiny smoke test")
print("=" * 64)

smoke_model = None
if not RUN_STAGE2:
    print("SKIPPED — VAE_STAGE2 is off.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TRAIN_DIR}.")
else:
    SMOKE_IMAGES = 256
    SMOKE_EPOCHS = 2

    smoke_ds, smoke_loader = make_loader(TRAIN_DIR, batch_size=32, shuffle=True, limit=SMOKE_IMAGES)
    print(f"smoke subset: {len(smoke_ds)} images, {len(smoke_loader)} batches/epoch")

    # Verify the pipeline output before feeding the model anything.
    xb, sb = next(iter(smoke_loader))
    print(f"batch shape : {tuple(xb.shape)}  dtype {xb.dtype}")
    print(f"value range : [{xb.min():.3f}, {xb.max():.3f}]")
    assert xb.shape[1:] == (1, IMG_SIZE, IMG_SIZE), f"unexpected batch shape {tuple(xb.shape)}"
    assert 0.0 <= xb.min() and xb.max() <= 1.0, "images are not scaled to [0,1]"

    smoke_model = VAE(img_size=IMG_SIZE, latent_dim=LATENT_DIM).to(DEVICE)
    smoke_opt = torch.optim.Adam(smoke_model.parameters(), lr=LR)

    smoke_losses = []
    for epoch in range(1, SMOKE_EPOCHS + 1):
        t, r, k = train_one_epoch(smoke_model, smoke_loader, smoke_opt, beta=1.0)
        smoke_losses.append(t)
        print(f"  epoch {epoch}: total {t:9.2f}  recon {r:9.2f}  kl {k:7.2f}")

    assert all(np.isfinite(smoke_losses)), f"loss went non-finite: {smoke_losses}"
    assert smoke_losses[-1] < smoke_losses[0], (
        f"loss did not decrease ({smoke_losses[0]:.2f} -> {smoke_losses[-1]:.2f}); "
        "check the optimiser and that zero_grad is being called")

    # --- input vs reconstruction ---
    smoke_model.eval()
    with torch.no_grad():
        x_show = xb[:8].to(DEVICE)
        recon_show, _, _ = smoke_model(x_show)

    # Interleave originals (top row) with reconstructions (bottom row).
    pair = torch.cat([x_show.cpu(), recon_show.cpu()], dim=0)
    show_grid(pair, nrow=2, path=os.path.join(OUT_DIR, "stage2_smoke_recon.png"),
              title="Stage 2: originals (top) vs reconstructions after 2 epochs (bottom)",
              figsize=(12, 3.5))

    print(f"\nSTAGE 2 PASSED — loss {smoke_losses[0]:.2f} -> {smoke_losses[-1]:.2f}, all finite.")

## Stage 3 — full training run

All 9,664 training images for `EPOCHS` epochs, timed, with the validation split evaluated each
epoch. We track the reconstruction and KL terms **separately** as well as the total, because the
total alone hides the failure mode that matters: if KL falls to near zero and stays there, the
model has posterior-collapsed and the manifold in stage 4 will be worthless, even though the
total loss looks like it is improving.

Adam rather than SGD here — VAEs are not the sharp-minima generalisation problem that image
classifiers are, and Adam's per-parameter step sizes handle the two-term loss more gracefully.

Gated behind `VAE_STAGE3=1`.

In [ ]:
print("=" * 64)
print("STAGE 3 — full training run")
print("=" * 64)

model = None
history = None

if not RUN_STAGE3:
    print("SKIPPED — set VAE_STAGE3=1 to run this stage.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TRAIN_DIR}.")
else:
    train_ds, train_loader = make_loader(TRAIN_DIR, BATCH_SIZE, shuffle=True)
    val_ds, val_loader = make_loader(VAL_DIR, BATCH_SIZE, shuffle=False)
    print(f"train: {len(train_ds):,} images / {len(train_loader)} batches")
    print(f"val  : {len(val_ds):,} images / {len(val_loader)} batches")

    model = VAE(img_size=IMG_SIZE, latent_dim=LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {k: [] for k in
               ("train_total", "train_recon", "train_kl",
                "val_total", "val_recon", "val_kl", "beta", "epoch_time")}

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()     # CUDA is async; sync or we time kernel launches, not work
    t_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        beta = beta_at_epoch(epoch)

        tr_t, tr_r, tr_k = train_one_epoch(model, train_loader, optimizer, beta)
        va_t, va_r, va_k = evaluate(model, val_loader, beta)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        for key, val in (("train_total", tr_t), ("train_recon", tr_r), ("train_kl", tr_k),
                         ("val_total", va_t), ("val_recon", va_r), ("val_kl", va_k),
                         ("beta", beta), ("epoch_time", dt)):
            history[key].append(val)

        # flush=True so the line appears in the SLURM log immediately, not at job end.
        print(f"epoch {epoch:3d}/{EPOCHS}  beta {beta:.2f}  "
              f"train {tr_t:8.2f} (rec {tr_r:8.2f}, kl {tr_k:6.2f})  "
              f"val {va_t:8.2f} (rec {va_r:8.2f}, kl {va_k:6.2f})  {dt:.1f}s", flush=True)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - t_start
    print(f"\ntrained in {total_time:.1f}s ({np.mean(history['epoch_time']):.1f}s/epoch)")

    # Single checkpoint only — home quota on Rangpur is 16GB. state_dict, not the whole
    # model object, so the file stays small and does not pickle the class definition.
    ckpt = os.path.join(OUT_DIR, "vae_oasis.pt")
    torch.save({"state_dict": model.state_dict(),
                "img_size": IMG_SIZE, "latent_dim": LATENT_DIM}, ckpt)
    print(f"saved checkpoint -> {ckpt} ({os.path.getsize(ckpt) / 1e6:.1f} MB)")

    with open(os.path.join(OUT_DIR, "vae_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # --- loss curves: total, reconstruction and KL separately ---
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    ep = range(1, EPOCHS + 1)

    axes[0].plot(ep, history["train_total"], label="train")
    axes[0].plot(ep, history["val_total"], "--", label="val")
    axes[0].set_title("Total loss (recon + beta*KL)")

    axes[1].plot(ep, history["train_recon"], label="train")
    axes[1].plot(ep, history["val_recon"], "--", label="val")
    axes[1].set_title("Reconstruction (BCE)")

    axes[2].plot(ep, history["train_kl"], label="train")
    axes[2].plot(ep, history["val_kl"], "--", label="val")
    ax_b = axes[2].twinx()
    ax_b.plot(ep, history["beta"], ":", color="grey", label="beta")
    ax_b.set_ylabel("beta", color="grey")
    axes[2].set_title("KL divergence  (watch for collapse to 0)")

    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(True, alpha=0.3); ax.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "stage3_loss_curves.png"), dpi=120)
    print(f"saved -> {os.path.join(OUT_DIR, 'stage3_loss_curves.png')}")
    plt.show()
    plt.close(fig)

    # Explicit collapse check — the failure the total loss hides.
    if history["train_kl"][-1] < 1e-2:
        print("\nWARNING: KL is ~0 — the model has posterior-collapsed. The manifold will be "
              "meaningless. Lower VAE_BETA or lengthen VAE_KL_WARMUP and retrain.")

## Stage 4 — manifold visualisation

**This is what the marks hinge on.** Both accepted forms are produced:

### (a) Sampling grid — a 2D image of the manifold

Sweep a 2D plane through latent space, decode every point, and tile the results. Walking across
the image should show the brain morphing smoothly, which is the visual proof that the KL term did
its job and the space is continuous rather than full of holes.

The grid coordinates are spaced by the **inverse CDF of the normal** rather than linearly. The
prior is `N(0, I)`, so linear spacing would oversample the dense centre and waste most of the
grid; quantile spacing gives each cell equal probability mass under the prior.

Which plane we sweep depends on `LATENT_DIM`:

- **`LATENT_DIM == 2`** — sweep the two raw axes directly. The classic textbook manifold.
- **`LATENT_DIM > 2`** — sweep the two directions of greatest variance in the *encoded training
  set* (found by PCA), holding the mean position on all other axes. Still a real 2D plane
  through the learned manifold, just chosen to capture the most structure.

### (b) Dimensionality-reduction scatter

Encode the test set, take each image's `mu`, project those vectors to 2D and scatter them. Points
are coloured by **slice index**, parsed from the filename — the VAE never saw that label, so if
the colours come out organised, the model has genuinely learned anatomical position from pixels
alone. That is a much stronger result to show a demonstrator than an unlabelled blob.

UMAP is preferred but is not in the base environment and the compute nodes have no internet, so
the code falls back to scikit-learn's t-SNE, then PCA, and states which one it used.

In [ ]:
print("=" * 64)
print("STAGE 4 — manifold visualisation")
print("=" * 64)

# Fall back to the stage-2 model if stage 3 did not run, so this stage can be exercised cheaply.
viz_model = model if model is not None else smoke_model

if not RUN_STAGE4:
    print("SKIPPED — set VAE_STAGE4=1 to run this stage.")
elif viz_model is None:
    print("SKIPPED — no trained model. Run stage 2 or 3, or load a checkpoint:")
    print("  ck = torch.load(os.path.join(OUT_DIR, 'vae_oasis.pt'))")
    print("  viz_model = VAE(ck['img_size'], ck['latent_dim']).to(DEVICE)")
    print("  viz_model.load_state_dict(ck['state_dict'])")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TEST_DIR}.")
else:
    from scipy.stats import norm

    viz_model.eval()

    # ---------------------------------------------------------------- encode the test set
    test_ds, test_loader = make_loader(TEST_DIR, BATCH_SIZE, shuffle=False)

    all_mu, all_slice = [], []
    with torch.no_grad():
        for x, s in test_loader:
            mu, _ = viz_model.encode(x.to(DEVICE))
            all_mu.append(mu.cpu())
            all_slice.append(s)
    all_mu = torch.cat(all_mu).numpy()          # (N, latent_dim)
    all_slice = torch.cat(all_slice).numpy()    # (N,)
    print(f"encoded {all_mu.shape[0]:,} test images -> latent {all_mu.shape[1]}D")

    # ---------------------------------------------------------------- (a) sampling grid
    GRID = 16
    # Equal-probability spacing under the N(0,1) prior. 0.005..0.995 avoids the infinite tails.
    quantiles = norm.ppf(np.linspace(0.005, 0.995, GRID))

    if LATENT_DIM == 2:
        # Direct sweep of the two raw latent axes.
        zs = np.zeros((GRID * GRID, 2), dtype=np.float32)
        for i, yi in enumerate(quantiles[::-1]):     # reversed so +y points up in the image
            for j, xj in enumerate(quantiles):
                zs[i * GRID + j] = (xj, yi)
        sweep_label = "latent axes z0 (x) and z1 (y)"
    else:
        # Sweep the plane of greatest variance in the encoded data.
        from sklearn.decomposition import PCA
        pca2 = PCA(n_components=2).fit(all_mu)
        centre = all_mu.mean(axis=0)
        # Scale each PCA direction by the actual spread of the data along it, so the sweep
        # covers the populated region rather than an arbitrary unit box.
        scale = all_mu @ pca2.components_.T
        s0, s1 = scale[:, 0].std(), scale[:, 1].std()

        zs = np.zeros((GRID * GRID, LATENT_DIM), dtype=np.float32)
        for i, yi in enumerate(quantiles[::-1]):
            for j, xj in enumerate(quantiles):
                zs[i * GRID + j] = (centre
                                    + xj * s0 * pca2.components_[0]
                                    + yi * s1 * pca2.components_[1])
        var = pca2.explained_variance_ratio_
        sweep_label = (f"top-2 PCA directions of the encoded set "
                       f"({100 * var.sum():.0f}% of latent variance)")

    with torch.no_grad():
        decoded = viz_model.decode(torch.from_numpy(zs).to(DEVICE)).cpu().numpy()

    # Tile the decoded images into one big canvas by direct array slicing — far faster than
    # GRID*GRID matplotlib subplots, and produces a single clean image.
    S = IMG_SIZE
    canvas = np.zeros((GRID * S, GRID * S), dtype=np.float32)
    for i in range(GRID):
        for j in range(GRID):
            canvas[i * S:(i + 1) * S, j * S:(j + 1) * S] = decoded[i * GRID + j, 0]

    plt.figure(figsize=(11, 11))
    plt.imshow(canvas, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.title(f"VAE manifold: {GRID}x{GRID} decoded sweep across {sweep_label}")
    plt.tight_layout()
    manifold_path = os.path.join(OUT_DIR, "stage4a_manifold_grid.png")
    plt.savefig(manifold_path, dpi=140, bbox_inches="tight")
    print(f"saved -> {manifold_path}")
    plt.show()
    plt.close()

    # ---------------------------------------------------------------- (b) 2D scatter
    method = None
    try:
        import umap                                        # noqa: F401
        reducer = umap.UMAP(n_components=2, random_state=SEED)
        emb = reducer.fit_transform(all_mu)
        method = "UMAP"
    except ImportError:
        # UMAP is not in the base env and compute nodes have no internet, so fall back.
        if LATENT_DIM > 2:
            from sklearn.manifold import TSNE
            # t-SNE is O(N^2)-ish; subsample if the test split is large.
            idx = np.random.RandomState(SEED).choice(
                len(all_mu), size=min(3000, len(all_mu)), replace=False)
            emb_sub = TSNE(n_components=2, random_state=SEED,
                           init="pca", perplexity=30).fit_transform(all_mu[idx])
            emb = np.full((len(all_mu), 2), np.nan)
            emb[idx] = emb_sub
            method = "t-SNE (sklearn; UMAP unavailable offline)"
        else:
            emb = all_mu
            method = "raw 2D latent (no reduction needed)"

    print(f"projection method: {method}")

    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=all_slice, cmap="viridis", s=4, alpha=0.6)
    fig.colorbar(sc, ax=ax, label="slice index (never seen during training)")
    ax.set_title(f"OASIS test set encoded to 2D via {method}\n"
                 f"colour = anatomical slice position")
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    scatter_path = os.path.join(OUT_DIR, "stage4b_latent_scatter.png")
    plt.savefig(scatter_path, dpi=130, bbox_inches="tight")
    print(f"saved -> {scatter_path}")
    plt.show()
    plt.close(fig)

## Stage 5 — save artifacts

Writes PNGs so a non-interactive SLURM run leaves something inspectable behind:

- **reconstructions** — real test images above, the model's reconstruction below
- **random samples** — `z ~ N(0, I)` decoded, i.e. brains the model invents from nothing.
  These test the prior directly: if they look like plausible brains, the KL term has genuinely
  matched the aggregate posterior to the prior.
- **an interpolation** — walk a straight line in latent space between two real encoded images.
  A smooth morph is strong evidence of a well-formed manifold; an abrupt jump halfway means the
  space still has holes.

In [ ]:
print("=" * 64)
print("STAGE 5 — save artifacts")
print("=" * 64)

art_model = model if model is not None else smoke_model

if not RUN_STAGE5:
    print("SKIPPED — set VAE_STAGE5=1 to run this stage.")
elif art_model is None:
    print("SKIPPED — no trained model available.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TEST_DIR}.")
else:
    art_model.eval()
    _, art_loader = make_loader(TEST_DIR, batch_size=8, shuffle=False)
    x_real, _ = next(iter(art_loader))
    x_real = x_real.to(DEVICE)

    with torch.no_grad():
        # --- reconstructions ---
        x_recon, _, _ = art_model(x_real)
        show_grid(torch.cat([x_real.cpu(), x_recon.cpu()]), nrow=2,
                  path=os.path.join(OUT_DIR, "stage5_reconstructions.png"),
                  title="Test images (top) and VAE reconstructions (bottom)",
                  figsize=(12, 3.5))

        # --- random samples from the prior ---
        z = torch.randn(16, LATENT_DIM, device=DEVICE)
        samples = art_model.decode(z)
        show_grid(samples.cpu(), nrow=2,
                  path=os.path.join(OUT_DIR, "stage5_samples.png"),
                  title="Generated samples: z ~ N(0, I) decoded",
                  figsize=(16, 4))

        # --- latent interpolation between two real images ---
        mu_a, _ = art_model.encode(x_real[0:1])
        mu_b, _ = art_model.encode(x_real[1:2])
        # Linear interpolation: z(t) = (1-t)*mu_a + t*mu_b for t from 0 to 1.
        ts = torch.linspace(0, 1, 10, device=DEVICE).view(-1, 1)
        z_interp = (1 - ts) * mu_a + ts * mu_b
        interp = art_model.decode(z_interp)
        show_grid(interp.cpu(), nrow=1,
                  path=os.path.join(OUT_DIR, "stage5_interpolation.png"),
                  title="Latent interpolation between two real test images",
                  figsize=(18, 2.2))

    print(f"\nartifacts written to {OUT_DIR}/")
    for f in sorted(os.listdir(OUT_DIR)):
        print(f"  {f}")

## Q&A — likely demonstrator questions

**What is the reparameterisation trick and why is it needed?**
We need to sample `z ~ N(mu, sigma^2)` and still backpropagate into the encoder, but sampling is
not a differentiable operation. The trick rewrites the sample as `z = mu + sigma * eps` with
`eps ~ N(0, I)`. The randomness now lives entirely in `eps`, which has no parameters and is a
constant to autograd, so `z` becomes a smooth differentiable function of `mu` and `sigma` while
keeping the distribution we wanted. Without it the encoder receives no gradient at all.

**Why does the encoder output `mu` and `logvar` instead of a single vector?**
Because a VAE maps each image to a *distribution*, not a point. That is the whole difference from
a plain autoencoder: overlapping distributions tile the latent space continuously, so points
between training images decode to something sensible. `log` variance specifically, because
variance must be positive — predicting its log means any real network output is valid, and the KL
formula wants `log(var)` anyway.

**What does the KL term actually do?**
It measures how far each image's encoded Gaussian has drifted from the `N(0, I)` prior and
penalises that distance, which pulls all the posteriors together into one well-shaped blob. It is
what makes sampling possible: without it the encoder would scatter images arbitrarily and shrink
every sigma to zero, giving a discontinuous space where `z ~ N(0, I)` decodes to noise.

**Why weight reconstruction against KL?**
They want opposite things — reconstruction wants every image in its own tight private region, KL
wants every image mapped to the same distribution. Too much KL gives posterior collapse: the
decoder ignores `z` and every sample is the same blurry average brain. Too little and the space
is full of holes, so reconstructions are sharp but the manifold plot is meaningless. `beta = 1`
is the true ELBO; above 1 is a beta-VAE, trading sharpness for smoother, more disentangled
latents.

**How do you know it has not posterior-collapsed?**
The KL curve in stage 3 is plotted separately for exactly this reason. If KL falls to ~0 and
stays there, the encoder is emitting the prior regardless of input. Stage 3 prints an explicit
warning if the final KL is below 0.01. The other tell is that random samples all look identical.

**Why BCE for reconstruction rather than MSE?**
The decoder ends in a sigmoid and the inputs are scaled to [0,1], so BCE is the matching
likelihood. It penalises confident errors more steeply than MSE, which in practice gives less
washed-out reconstructions. MSE would also work — it corresponds to assuming Gaussian rather than
Bernoulli pixel likelihoods.

**Why sum over pixels but average over the batch?**
The KL term is intrinsically a per-image sum over latent dimensions. If reconstruction were a
per-pixel mean, the two terms would differ by a factor of ~4096 and `beta = 1` would not
correspond to the actual variational bound. Summing pixels keeps them commensurate.

**Why downsample to 64×64?**
256×256 is 16× the pixels for little modelling benefit on registered, smoothly-varying brain
slices, and it forces a smaller batch or a narrower network. `IMG_SIZE` is a config value —
set `VAE_IMG=128` if you want sharper reconstructions and have the GPU budget.

**Why no data augmentation, when part 3.2 used it heavily?**
The OASIS slices are spatially registered. Flipping or rotating them would break the anatomical
alignment that makes the latent space interpretable. CIFAR-10 photos have no such canonical
orientation, so augmentation there is free extra data; here it would be actively harmful.

**What does the manifold plot actually show?**
The grid decodes a 2D plane swept through latent space, so adjacent cells differ by a small step
in `z`. Smooth morphing across the grid demonstrates the space is continuous. The scatter shows
where real test images land, coloured by slice index — a label the model never saw. Organised
colour means the VAE recovered anatomical position from pixels alone.

**Why is the KL warm-up there?**
Early on the decoder is useless, so the cheapest way to reduce the loss is to collapse KL to
zero — and models that start collapsed rarely recover. Ramping `beta` from 0 over the first 10
epochs lets the decoder become useful before the regulariser starts to bite.